# Notebook 

## RL question game

In [120]:
import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np

# from typing import Optional
import gymnasium as gym

from torch.distributions import Categorical

### Simple version: finite W

In [149]:
class Sherlock_simple_env(gym.Env) : 
    def __init__(self, props : int = 8) -> None:
        self._size = 2 ** props

        self._S : np.ndarray
        # self._context : tuple[np.ndarray] 
        self._target : int 

        self.action_space = gym.spaces.MultiBinary(self._size)

        self.observation_space = gym.spaces.Dict(
            {
            "S": gym.spaces.MultiBinary(self._size),
            # "context" : gym.spaces.Sequence(gym.spaces.MultiBinary(self._size))
            }
        )

    def get_observation(self) -> dict :
        # return {"S": self._S, "context" : self._context}
        return {"S" : self._S}
    
    def get_info(self) -> dict :
        return {"closeness" : 1 - sum(self._S)/self._size}

    def reset(self, seed: int | None = None, options: dict | None = None) -> tuple[dict, dict] :
        super().reset(seed=seed)
        
        self._S = np.ones(self._size, dtype=int)
        # self._context = (np.ones(self._size, dtype=int), )
        self._target = self.np_random.integers(1, self._size)

        observation = self.get_observation()
        info = self.get_info()

        return observation, info
    
    def step(self, action: np.ndarray) -> tuple[dict, int, bool, bool, dict] :

        # Should make sure to not be able to choose worlds that have been eliminated
        if action[self._target] :
            self._S = self._S & action 
        else : 
            self._S = self._S & (~ action)

        # self._context.append
        fini = sum(self._S) == 1

        truncated = False 

        # Perhaps punish for not gaining new info
        # reward = 1 if fini else (- diff_info)
        reward = 1 if fini else 0

        observation = self.get_observation()
        info = self.get_info()
        return observation, reward, fini, truncated, info

In [174]:
e = Sherlock_simple_env(props= 5)
e.reset()
fini = False 

while not fini : 
    action = e.action_space.sample()
    obs, _, fini, _, info = e.step(action)
    print(action)
    print(sum(action))
    print(info)

print(obs)


[1 1 0 0 1 1 1 0 1 1 0 1 1 0 0 1 1 1 0 1 1 1 0 1 1 1 0 0 0 0 1 1]
20
{'closeness': np.float64(0.375)}
[1 1 1 1 0 0 1 1 1 0 1 1 1 0 1 1 1 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0]
15
{'closeness': np.float64(0.6875)}
[1 0 1 1 0 1 0 0 0 0 1 0 0 0 0 1 1 0 0 1 0 0 1 0 0 1 1 1 0 0 1 0]
13
{'closeness': np.float64(0.78125)}
[0 0 1 0 0 1 1 0 0 0 0 0 1 0 1 1 0 0 0 0 0 1 1 1 1 0 0 0 0 1 1 0]
12
{'closeness': np.float64(0.84375)}
[1 0 1 0 0 0 0 1 0 0 1 1 0 0 1 1 1 1 1 0 1 1 0 0 1 1 0 0 0 0 1 1]
16
{'closeness': np.float64(0.90625)}
[0 1 1 1 0 1 0 1 0 0 0 0 0 1 1 0 0 0 0 0 0 1 0 1 0 1 1 1 0 0 0 1]
13
{'closeness': np.float64(0.9375)}
[0 1 0 1 1 0 0 1 0 0 1 1 1 1 1 0 1 0 0 0 0 1 0 0 1 1 1 0 0 1 1 1]
17
{'closeness': np.float64(0.9375)}
[0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 1 1 0 0 0 1 1 0 0 0 0 0 0 0 0]
6
{'closeness': np.float64(0.96875)}
{'S': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0])}


In [113]:
a = np.array([1,1,1])
b = np.array([0,0,1])
a & (~ b)

array([1, 1, 0])

In [118]:
action_space = gym.spaces.Box(3, 17, shape=(2,))
r = (action_space.sample(),)
r

(array([6.753811 , 4.2201343], dtype=float32),)

### Harder version

In [18]:
class Game() :

    def __init__(self, answer : list[tuple[int, int]], context : list[tuple[int, int]] = []) -> None:
        self._answer = answer 
        self._context = context

    def is_solved(self) -> bool :
        return False

def generate_game() -> Game :
    return Game([(0,0)], [])

class Sherlock_Env() :

    def __init__(self) -> None :
        pass

    def _step(self, action : torch.Tensor) -> None :
        pass 

    def _reset(self) -> None :
        pass

class Sherlock(nn.Module) : 

    def __init__(self) -> None:
        super().__init__()


In [19]:
def train(env : Sherlock_Env,
          sherlock : Sherlock,
          optimizer: optim.Optimizer,
          episodes : int = 1000 ) -> None :
    
    for game in range(episodes) : 
        state = env._reset() 
        rewards = []
        fini = False 

        while not fini : 

            # pick action (guess) 
            probs = sherlock(state)
            d = Categorical(probs) 
            action = d.sample()

            # do action and get rewards
            state, reward, fini, _ = env._step(action.item())
            rewards.append(reward)
            
            # after every run, update model
            if fini : 
                break
